# Exploratory analysis of saved evaluator trajectories

## Goal

Inspect an existing experiment without making model calls. This companion notebook is secondary: the complete pipeline runs from `scripts/`. No research findings are prefilled and no outputs are saved in this template. Mock results validate software behavior only.

## Setup

From the repository root, install `pip install -e ".[analysis]"` and a Jupyter frontend, then run `python scripts/run_experiment.py --config configs/experiment.yaml` to create `results/latest.csv`. Change the input path below to a real experiment when available. Run cells from top to bottom.


In [ ]:
from pathlib import Path
import sys

ROOT = next((path for path in (Path.cwd(), Path.cwd().parent)
             if (path / "pyproject.toml").is_file() and (path / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Launch the notebook from the repository root or its notebooks directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.serialization import read_completed_results
from src.analysis.pipeline import validate_trajectories
from src.analysis.statistics import summarize_condition, paired_endpoint_comparisons
from src.metrics.core import trajectory_metrics

RESULTS = ROOT / "results" / "latest.csv"
SEED = 42
BOOTSTRAP_SAMPLES = 1000


## Steps

### 1. Load and validate saved records

The input must contain one explicitly labeled experiment with complete task × condition × round trajectories. Hidden test contents are neither loaded separately nor displayed here. Read the matching run manifest for the exact models, seeds, dataset digest and execution backend.


In [ ]:
rows = validate_trajectories(read_completed_results(RESULTS))
run_kind = rows[0]["run_kind"]
experiment_id = rows[0]["experiment_id"]
final_round = max(row["round"] for row in rows)
conditions = sorted({row["condition"] for row in rows})
print({"experiment_id": experiment_id, "run_kind": run_kind,
       "tasks": len({row["task_id"] for row in rows}), "final_round": final_round,
       "input": str(RESULTS)})
if run_kind == "mock":
    print("MOCK VALIDATION ONLY — empirical hypotheses remain untested.")


### 2. Recompute final-round estimates

J predicts the binary event that all hidden test units pass; G is the fraction of units passed. Therefore J − G is a diagnostic of evaluation mismatch, while Brier and ECE measure calibration against the binary event. False approval means P(approve | at least one hidden test unit fails). Intervals use a task-cluster percentile bootstrap, retaining paired conditions and rounds. Undefined denominators stay missing.


In [ ]:
summary = [
    {"condition": condition, **summarize_condition(
        [row for row in rows if row["condition"] == condition and row["round"] == final_round],
        samples=BOOTSTRAP_SAMPLES, seed=SEED)}
    for condition in conditions
]
for entry in summary:
    print({key: (round(entry[key], 4) if isinstance(entry[key], float) else entry[key])
           for key in ("condition", "n_tasks", "mean_judge_score", "mean_ground_truth",
                       "mean_gap", "false_approval_rate", "brier_score", "ece")})


### 3. Plot mean gaps with task-bootstrap intervals

Final candidates can differ across feedback conditions, so these comparisons describe full feedback policies. They do not isolate how different judges score a fixed candidate. Points and intervals are taken directly from the estimates above.


In [ ]:
import os
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from src.analysis.figures import COLORS, LABELS, MARKERS

with plt.rc_context({"font.family": "DejaVu Sans", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False}):
    fig, ax = plt.subplots(figsize=(7.2, 4.6))
    for index, entry in enumerate(summary):
        condition = entry["condition"]
        color = COLORS.get(condition, "#606A73")
        low, high = entry["mean_gap_ci_low"], entry["mean_gap_ci_high"]
        if low is not None:
            ax.hlines(index, low, high, color=color, linewidth=1.6)
        ax.scatter(entry["mean_gap"], index, color=color,
                   marker=MARKERS.get(condition, "o"), s=50, zorder=3)
    ax.axvline(0, color="#454A50", linewidth=1)
    ax.set_yticks(range(len(summary)), [LABELS.get(entry["condition"], entry["condition"])
                                      for entry in summary])
    ax.invert_yaxis()
    ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{100 * value:.0f}"))
    ax.set_xlabel("Mean J − G (percentage points); 95% task-bootstrap CI")
    ax.set_title(("MOCK VALIDATION · " if run_kind == "mock" else "")
                 + f"Final-round evaluation gaps (round {final_round})")
    ax.grid(axis="x", color="#E1E4E8", linewidth=.6)
    fig.tight_layout()
    plt.show()


### 4. Inspect paired changes and the no-feedback control

H1 concerns changes in J and G, not the final score alone. These endpoint metrics compare round 0 with T. When `baseline_no_feedback` is configured, the paired difference of changes helps distinguish evaluator-feedback optimization from repeated revision. All such contrasts remain exploratory; task resampling does not cover variation across independently trained models.


In [ ]:
trajectory_summary = trajectory_metrics(rows)
for entry in trajectory_summary:
    print(entry)
endpoint_comparisons = paired_endpoint_comparisons(rows, samples=BOOTSTRAP_SAMPLES, seed=SEED)
for entry in endpoint_comparisons:
    if entry["metric"] == "delta_gap" and entry["reference"] == "baseline_no_feedback":
        print(entry)


## Checks

The validation step rejects duplicated records and incomplete trajectory grids. Calibration targets the all-tests-pass event. No conclusion should be drawn from mock measurements. Within-ensemble score variance is member disagreement, not the sampling variance of the ensemble mean. High-gap case selection and all primary figures are reproducible through `python scripts/generate_report.py --results results/latest.csv`.

## Next steps

For real runs, inspect paired intervals and manually review high-gap failures before writing conclusions. Report task sampling, model configuration, run kind, benchmark contamination risks and execution limitations with every empirical claim. Use fixed-candidate rescoring, independent seeds and cross-family evaluators to investigate mechanisms.
